In [5]:
#simple linear regression
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F

x = torch.arange(1, 100, 0.1).unsqueeze(1) #[B, 1]
y = x ** 2

dataset = TensorDataset(x, y)

loader = DataLoader(dataset, batch_size=10, shuffle=True)

class regression_model(nn.Module):
    def __init__(self, x_mean, x_std, y_mean, y_std):
        super().__init__()
        self.register_buffer("x_mean", x_mean)
        self.register_buffer("x_std",  x_std)
        self.register_buffer("y_mean", y_mean)
        self.register_buffer("y_std",  y_std)

        self.h = nn.Linear(in_features=1, out_features=2)
        self.lh = nn.Linear(in_features=2, out_features=1)

    def forward(self, x):
        x_n = (x - self.x_mean) / self.x_std
        return F.sigmoid(self.lh(self.h(x_n)))         

    def denorm(self, z):                         # normalized -> physical units
        return z * self.y_std + self.y_mean
        
reg_model = regression_model(x.mean(), x.std(), y.mean(), y.std())

opt = torch.optim.SGD(reg_model.parameters(), lr=0.01)

for epoch in range(10):   

    epoch_loss = 0.0
    
    for xb, yb in loader:
        
        opt.zero_grad()
        
        predictions = reg_model(xb) #normalized inside the model
        yb_n = (yb - reg_model.y_mean) / reg_model.y_std #normalize the output

        loss = F.mse_loss(predictions, yb_n)

        loss.backward()

        opt.step()

        epoch_loss += loss.item() * xb.size(0)

    print(f"Epoch [{epoch+1}/100], Loss: {epoch_loss/len(dataset):.6f}")

# inference: raw x in, physical y out
with torch.no_grad():
    print(reg_model.denorm(reg_model(torch.tensor([[10.0], [50.0]]))))

Epoch [1/100], Loss: 1.342259
Epoch [2/100], Loss: 1.068559
Epoch [3/100], Loss: 0.905165
Epoch [4/100], Loss: 0.771067
Epoch [5/100], Loss: 0.670278
Epoch [6/100], Loss: 0.610121
Epoch [7/100], Loss: 0.576804
Epoch [8/100], Loss: 0.557051
Epoch [9/100], Loss: 0.544164
Epoch [10/100], Loss: 0.535042
tensor([[3368.9026],
        [3794.6101]])


In [9]:
print(x.mean(), x.std(), (x**2).mean())

H = 2*torch.tensor([[(x**2).mean(), x.mean()], [x.mean(), 1.0]])
print(H)
lmin, lmax = torch.linalg.eigvalsh(H)
print("lmin:", lmin, "lmax:", lmax)

tensor(50.4500) tensor(28.5933) tensor(3361.9514)
tensor([[6.7239e+03, 1.0090e+02],
        [1.0090e+02, 2.0000e+00]])
lmin: tensor(0.4858) lmax: tensor(6725.4170)
